In [9]:
# Import Library yang dibutuhkan
import pickle # menggunakan pickle untuk load data
import sys # menggunakan sys untuk mengatasi error pada versi pandas terbaru
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df_reliable = pd.read_pickle('../data/wm811k_v3_reliable.pkl')
print(f"Loaded: {df_reliable.shape[0]} baris, {df_reliable.shape[1]} kolom")

Loaded: 809424 baris, 25 kolom


In [10]:
# mari kita validasi struktur data sebelum mulai eksplorasi lebih lanjut
try:
    print(f"Total baris di df_reliable: {len(df_reliable)}")
except NameError:
    print("ERROR: df_reliable tidak ditemukan! Kernel mungkin di-restart.")
    print("Solusi: Run ulang cell di Day 12 yang membuat df_reliable.")

Total baris di df_reliable: 809424


In [11]:
# apakah failure_type_clean ada dan sesuai
try:
    print("Kategori failureType_clean:")
    print(df_reliable['failureType_clean'].value_counts())
except KeyError:
    print("ERROR: Kolom 'failureType_clean' tidak ada di df_reliable!")
    print("Solusi: Cek nama kolom kamu di Day 12, mungkin namanya 'failureType' atau 'failure_Type'.")

Kategori failureType_clean:
failureType_clean
none         783905
Edge-Ring      9680
Edge-Loc       5189
Center         4294
Loc            3593
Scratch        1193
Random          866
Donut           555
Near-full       149
Name: count, dtype: int64


In [12]:
# apakah 4 kolom metric numerik sudah ada
required_cols = ['density_global', 'density_center', 'density_edge', 'edge_center_diff']
available_cols = [c for c in df_reliable.columns if 'density' in c or 'edge_center' in c]
print(f"Kolom density/edge yang tersedia: {available_cols}")

Kolom density/edge yang tersedia: ['density_global_reliable', 'density_regional_reliable', 'density_global', 'density_center', 'density_edge', 'edge_center_diff']


In [13]:
# Cek apakah ada yang hilang dari requirement kita
missing_cols = [col for col in required_cols if col not in available_cols]
if missing_cols:
    print(f"\nWARNING: Kolom ini TIDAK DITEMUKAN: {missing_cols}")
    print("Solusi: Kamu harus kembali ke proses feature engineering Day 12 dan buat kolom ini.")
else:
    print("\nSEMUA AMAN! 4 kolom metric numerik siap dipakai untuk pivot_table.")


SEMUA AMAN! 4 kolom metric numerik siap dipakai untuk pivot_table.


In [15]:
summary_pivot = pd.pivot_table(
    df_reliable,
    index='failureType_clean',
    values=['density_global', 'density_center', 'density_edge', 'edge_center_diff'],
    aggfunc=['mean', 'count'],
    margins=True,
    margins_name='All'
)
print(summary_pivot.round(3))

                            mean                                               \
                  density_center density_edge density_global edge_center_diff   
failureType_clean                                                               
Center                     0.243        0.221          0.232           -0.022   
Donut                      0.367        0.172          0.277           -0.195   
Edge-Loc                   0.124        0.257          0.183            0.133   
Edge-Ring                  0.059        0.258          0.151            0.199   
Loc                        0.145        0.156          0.150            0.012   
Near-full                  0.898        0.852          0.877           -0.046   
Random                     0.468        0.493          0.481            0.025   
Scratch                    0.081        0.125          0.101            0.043   
none                       0.072        0.118          0.093            0.046   
All                        0

In [18]:
# Langkah 3: Sanity Check

# 1. Re-create diff_stats (karena di notebook baru variabel ini belum ada)
# Ini menghitung statistik deskriptif edge_center_diff menggunakan groupby langsung
diff_stats = df_reliable.groupby('failureType_clean')['edge_center_diff'].describe()

print("=== 1. Perbandingan Mean edge_center_diff ===")
print("Dari Pivot Table:")
print(summary_pivot['mean']['edge_center_diff'])
print("\nDari GroupBy Describe (diff_stats):")
print(diff_stats['mean'])

print("\n===========================================\n")

# 2. Cek total count
print("=== 2. Validasi Total Count ===")
total_count_pivot = summary_pivot['count']['density_global']['All']
total_baris_data = len(df_reliable)

print(f"Total count dari Pivot Table (baris 'All'): {total_count_pivot}")
print(f"Total baris df_reliable (len): {total_baris_data}")

# Cek apakah identik
match_count = total_count_pivot == total_baris_data
print(f"Apakah total count match? {match_count}")

# Cek apakah mean edge_center_diff match
# Kita buang baris 'All' dulu dari pivot agar bisa dibandingkan dengan diff_stats
mean_pivot_clean = summary_pivot['mean']['edge_center_diff'].drop('All')
mean_groupby_clean = diff_stats['mean']

# Sort index biar urutannya sama sebelum dibandingkan
mean_pivot_sorted = mean_pivot_clean.sort_index()
mean_groupby_sorted = mean_groupby_clean.sort_index()

# Cek selisihnya, kalau 0 berarti identik
selisih = (mean_pivot_sorted - mean_groupby_sorted).sum()
print(f"Apakah mean edge_center_diff match? (selisih harus 0): {selisih < 1e-9}")

=== 1. Perbandingan Mean edge_center_diff ===
Dari Pivot Table:
failureType_clean
Center      -0.022341
Donut       -0.194776
Edge-Loc     0.132926
Edge-Ring    0.199250
Loc          0.011851
Near-full   -0.045780
Random       0.024610
Scratch      0.043402
none         0.045970
All          0.047638
Name: edge_center_diff, dtype: float64

Dari GroupBy Describe (diff_stats):
failureType_clean
Center      -0.022341
Donut       -0.194776
Edge-Loc     0.132926
Edge-Ring    0.199250
Loc          0.011851
Near-full   -0.045780
Random       0.024610
Scratch      0.043402
none         0.045970
Name: mean, dtype: float64


=== 2. Validasi Total Count ===
Total count dari Pivot Table (baris 'All'): 809424
Total baris df_reliable (len): 809424
Apakah total count match? True
Apakah mean edge_center_diff match? (selisih harus 0): True


In [ ]:
size_interaction = pd.pivot_table(
    df_reliable,
    index='failureType_clean',
    columns='dieSize_binned',
    values='density_global',
    aggfunc='mean',
    observed=True
)
print(size_interaction.round(3))

# hitung juga count-nya, biar tau kalau ada sel yang isinya cuma segelintir baris (mean-nya gak reliable)
size_interaction_count = pd.pivot_table(
    df_reliable,
    index='failureType_clean',
    columns='dieSize_binned',
    values='density_global',
    aggfunc='count',
    observed=True
)
print("\nJumlah baris per sel:")
print(size_interaction_count)

dieSize_binned     kecil  sedang  besar
failureType_clean                      
Center             0.275   0.187  0.063
Donut              0.304   0.269  0.350
Edge-Loc           0.241   0.167  0.116
Edge-Ring          0.187   0.190  0.117
Loc                0.199   0.148  0.058
Near-full          0.886   0.869  0.916
Random             0.498   0.478  0.429
Scratch            0.145   0.107  0.047
none               0.138   0.092  0.045

Jumlah baris per sel:
dieSize_binned      kecil  sedang   besar
failureType_clean                        
Center               2521    1554     219
Donut                  17     490      48
Edge-Loc             1480    3211     498
Edge-Ring             357    4176    5147
Loc                   919    2239     435
Near-full              64      83       2
Random                212     618      36
Scratch               249     655     289
none               214989  372734  196182


In [19]:
def top_n_density(group, n=3):
    """
    Ambil n baris dengan density_global tertinggi dari satu grup.
    Aman lewat groupby().apply() karena tidak pernah mereferensikan
    kolom grouping (failureType_clean) di dalam body fungsi.
    """
    return group.nlargest(n, 'density_global')

# Simpan kolom yang relevan buat kandidat visual Day 14
# (waferMap ikut disimpan supaya bisa langsung diplot nanti,
#  tanpa perlu query ulang ke df_reliable)
cols_needed = ['lotName', 'waferIndex', 'waferMap',
               'density_global', 'density_center', 'density_edge',
               'edge_center_diff']

top3_per_category = df_reliable[cols_needed].groupby(
    df_reliable['failureType_clean'], observed=True
).apply(top_n_density, include_groups=False)

print(top3_per_category[['density_global', 'density_center', 'density_edge', 'edge_center_diff']])

                          density_global  density_center  density_edge  \
failureType_clean                                                        
Center            573           0.727142        0.961776      0.397924   
                  1097          0.717783        0.967941      0.366782   
                  1959          0.706983        0.964242      0.346021   
Donut             4674          0.678411        0.792593      0.561457   
                  4640          0.651032        0.641221      0.660517   
                  4689          0.637524        0.779555      0.457701   
Edge-Loc          6035          0.746124        0.797872      0.683761   
                  7851          0.719493        0.713996      0.725469   
                  7533          0.719444        0.753311      0.676471   
Edge-Ring         13835         0.451005        0.289786      0.632000   
                  17890         0.437186        0.282660      0.610667   
                  15906         0.4321